# SHIELD surface-limited analysis — K_d and K_r

When surface kinetics — not bulk diffusion — throttle permeation
(**surface-limited regime**, verified by flux ∝ P in
[`regime_analysis.ipynb`](regime_analysis.ipynb)), the concentration profile
across the sample is nearly flat, and the steady-state balance between
dissociative uptake and recombinative release reduces to

$$J = \tfrac{1}{2} K_d \, P \qquad \text{(identical surfaces, vacuum downstream)}$$

half of the atoms dissociated upstream leave through the downstream face.
A surface-limited run therefore measures the **dissociation coefficient**

$$K_d = 2J/P$$

and, through detailed balance with the Sieverts solubility K_s (which
guarantees the surface kinetics reproduce Sieverts' equilibrium), the
**recombination coefficient**

$$K_r = K_d / K_s^2$$

What a surface-limited run can **not** give: permeability, diffusivity, or
solubility. In particular the time-lag method fails — the breakthrough delay
measures surface kinetics, not e²/(6D).

**Read the numbers with the right confidence.** K_d is an *effective*
coefficient for this sample's surface state (oxide, roughness):

- the symmetric-surface factor ½ is exact only if both faces behave the same —
  asymmetry shifts K_d within a factor of ~2;
- any residual diffusion resistance makes 2J/P an underestimate;
- K_r inherits all of that **plus** the K_s uncertainty squared — treat it as
  an order-of-magnitude result;
- published coefficients differ by factor-of-2 unit conventions
  (per-molecule vs per-atom). This toolbox uses the atomic-flux convention
  J = K_d·P.

Prerequisite: the environment from the README's *Local development setup*
(`uv sync`, plus `uv pip install -e ../SHIELD-Data` for `shield_data`).

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
from uncertainties import ufloat

from shield_toolbox import fetch_run, load_results, process_run
from shield_toolbox.analysis import dissociation_coeff, recombination_coeff
from shield_toolbox.constants import N_A, TORR_TO_PA

## 1. The runs

From the ~550 K carbon-steel pressure sweep of [`regime_analysis.ipynb`](regime_analysis.ipynb):
the three runs below 50 Torr fitted a pressure exponent ≈ 1 — surface-limited —
and give K_d. The four runs furthest into the diffusion-limited side
(≥ 300 Torr) are processed too, **not** for K_d but to supply the solubility
K_s that detailed balance needs: K_s cannot be measured from surface-limited
data.

In [ ]:
SL_RUN_IDS = [
    "26.01.23_run_1_09h57",  # 41 Torr
    "26.01.28_run_1_10h52",  # 38 Torr
    "26.02.03_run_1_14h56",  # 24 Torr
]
DL_RUN_IDS = [
    "26.01.18_run_1_14h05",  # 400 Torr
    "26.01.18_run_2_20h10",  # 396 Torr
    "26.01.20_run_1_15h19",  # 573 Torr
    "26.01.27_run_1_16h43",  # 378 Torr
]

output_dir = Path(tempfile.mkdtemp())
for run_id in SL_RUN_IDS + DL_RUN_IDS:
    process_run(fetch_run(run_id)).write(output_dir)

results = load_results(output_dir, substrate="carbon steel", coating="none")

# Model-free steady flux, inverted from the apparent DL permeability Φ = J·e/√P.
p_up_pa = results.upstream_torr * TORR_TO_PA
results["flux"] = results.permeability * np.sqrt(p_up_pa) / results.thickness_m
results["flux_err"] = results.permeability_err * np.sqrt(p_up_pa) / results.thickness_m

results[["run_id", "temperature_K", "upstream_torr", "flux"]]

## 2. K_d from the surface-limited runs

One value per run; if the extraction is self-consistent they should agree
within error, since K_d is a property of the surface, not of the pressure.

In [ ]:
sl = results[results.run_id.isin(SL_RUN_IDS)]

k_d_runs = []
for row in sl.itertuples():
    flux = ufloat(row.flux, row.flux_err)
    k_d_run = dissociation_coeff(flux, row.upstream_torr * TORR_TO_PA)
    k_d_runs.append(k_d_run)
    print(
        f"{row.run_id}  P = {row.upstream_torr:5.1f} Torr  K_d = {k_d_run:.2e}  H/(m²·s·Pa)"
    )

k_d = sum(k_d_runs) / len(k_d_runs)
print(f"\nmean K_d = {k_d:.2e}  H/(m²·s·Pa)")

## 3. K_s from the diffusion-limited side, then K_r

S = Φ/D is only measurable in the DL regime. The run-to-run scatter of the
time-lag diffusivity dominates the per-run error bars, so the four DL runs
are combined as mean ± standard deviation rather than trusting any single
error bar. K_s enters K_r squared — this is where the order-of-magnitude
character of K_r comes from.

In [ ]:
dl = results[results.run_id.isin(DL_RUN_IDS)]
solubilities = dl.solubility.dropna()
k_s = ufloat(solubilities.mean(), solubilities.std())
print(f"K_s = {k_s:.2e}  H/(m³·Pa^0.5)   (from {len(solubilities)} DL runs)")

k_r = recombination_coeff(k_d, k_s)
print(f"K_r = {k_r:.1e}  m⁴/(H·s)")
print(f"    = {k_r * N_A:.1e}  m⁴/(mol·s)")

## 4. Sanity check: sticking probability

K_d has a hard physical ceiling: dissociation cannot outrun the rate at which
H₂ molecules hit the surface. Kinetic theory gives the impingement flux per
unit pressure as 1/√(2π·m·k_B·T), so the sticking probability implied by the
measured K_d,

$$\sigma = \tfrac{1}{2} K_d \sqrt{2\pi m_{H_2} k_B T}$$

must come out ≤ 1 — and for an air-formed oxide on steel, typically in the
10⁻⁹–10⁻⁶ range.

In [ ]:
K_B = 1.380649e-23  # J/K
M_H2 = 3.347e-27  # kg
temperature_K = sl.temperature_K.mean()

sigma = k_d * np.sqrt(2 * np.pi * M_H2 * K_B * temperature_K) / 2
print(f"sticking probability at {temperature_K:.0f} K : {sigma:.1e}")

## Where to go next

- The regime verdict these extractions rely on:
  [`regime_analysis.ipynb`](regime_analysis.ipynb).
- The diffusion-limited pipeline (Φ, D, S, Arrhenius):
  [`diffusion_limited_analysis.ipynb`](diffusion_limited_analysis.ipynb).
- A coating changes both the crossover pressure and the surface
  coefficients — re-run the pressure sweep after coating before extracting
  anything.